# 00 — Historical Backfill (run once)

Loads NOAA HURDAT2 Atlantic hurricane records (1851–present) and ~30 years of
EIA Gulf Coast gas prices into the bronze and silver Delta layers.

Run this notebook **once** before the first training run.  
The live polling jobs (01_bronze_ingest) continue to run on their normal schedule afterwards;
silver dedup prevents any overlap from being double-counted.

In [ ]:
%pip install httpx python-dotenv --quiet

In [ ]:
import subprocess, sys, os

CLONE_DIR = '/tmp/RainCheck'
if not os.path.isdir(os.path.join(CLONE_DIR, 'hurricane-gas-predictor', 'src')):
    subprocess.run(
        ['git', 'clone', '--depth=1', '--quiet',
         'https://github.com/Dlux2015/RainCheck.git', CLONE_DIR],
        check=True
    )
sys.path.insert(0, os.path.join(CLONE_DIR, 'hurricane-gas-predictor'))

os.environ['EIA_API_KEY']  = dbutils.secrets.get('raincheck', 'EIA_API_KEY')
os.environ['SUPABASE_URL'] = dbutils.secrets.get('raincheck', 'SUPABASE_URL')
os.environ['SUPABASE_KEY'] = dbutils.secrets.get('raincheck', 'SUPABASE_KEY')

In [ ]:
# Step 1 — HURDAT2 bronze (full Atlantic basin history, 1851–present)
from src.ingestion.hurdat2_ingest import run as ingest_hurdat2
ingest_hurdat2()

In [ ]:
# Step 2 — Extended EIA gas prices (~30 years, 1600 weeks)
# Drop the existing bronze table first so it's recreated with the correct
# DoubleType schema — avoids a Delta merge-fields conflict on price_usd.
# The backfill re-ingests 30 years of history so nothing is lost.
dbutils.fs.rm(
    '/Volumes/workspace/default/raincheck/delta/bronze/gas_prices',
    recurse=True
)

from src.ingestion.gas_price_ingest import run as ingest_gas
ingest_gas(weeks=1600)

In [ ]:
# Step 3 — Promote both to silver
from src.transforms.bronze_to_silver import clean_hurdat2_storms, clean_gas_prices
clean_hurdat2_storms(spark)
clean_gas_prices(spark)  # re-runs to pick up the extended price history

In [ ]:
# Step 4 — Rebuild gold features (now includes historical storms + prices)
from src.transforms.feature_engineering import run as build_features
build_features(spark)